# MBE Dynamics in a Rollout
Inspect how Matrix-Based Entropy (MBE) evolves as the model generates tokens.
For each hidden layer, we compute MBE on prefixes of increasing length,
revealing how representation diversity builds up during chain-of-thought.

**Key questions:**
1. Does MBE increase monotonically as more tokens are generated?
2. Which layers show the sharpest MBE dynamics?
3. Do correct vs incorrect rollouts differ in their MBE trajectory?

In [1]:
import os
import sys
import re
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

sys.path.insert(0, os.path.abspath("."))
from src.mbe import OnlineMBE, mbe_reverse_gram

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Load Model & Data
Load a Qwen3 model and a few GSM8K examples. Adjust `MODEL_NAME` to point at a checkpoint if available.

In [ ]:
# ---- Configuration ----
MODEL_NAME = "Qwen/Qwen3-0.6B"   # swap for a checkpoint path if available
MAX_NEW_TOKENS = 512
N_SAMPLES = 6                     # number of GSM8K problems to rollout
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# vLLM: set to True to use vLLM offline engine for fast batch generation.
# Generation uses vLLM; hidden-state extraction still uses the HF model.
USE_VLLM = True
VLLM_GPU_MEMORY_UTILIZATION = 0.5  # fraction of GPU VRAM for vLLM

# ---- Load HF model (for hidden-state extraction) ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map=DEVICE
)
model.eval()
print(f"HF Model: {MODEL_NAME}  |  Layers: {model.config.num_hidden_layers}  |  Device: {DEVICE}")

# ---- Optionally load vLLM offline engine (for fast generation) ----
vllm_engine = None
if USE_VLLM:
    try:
        from vllm import LLM, SamplingParams
        vllm_engine = LLM(
            model=MODEL_NAME,
            gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
            dtype="bfloat16",
            max_model_len=MAX_NEW_TOKENS + 512,  # prompt + completion headroom
        )
        print(f"vLLM engine loaded ({VLLM_GPU_MEMORY_UTILIZATION:.0%} VRAM)")
    except Exception as e:
        print(f"vLLM not available ({e}), falling back to HF generate")
        USE_VLLM = False

# ---- Load GSM8K test set ----
dataset = load_dataset("openai/gsm8k", "main")["test"]
print(f"GSM8K test set: {len(dataset)} examples")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B  |  Layers: 28  |  Device: cpu


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test set: 1319 examples


## 2. Helper Functions
Generate a rollout and extract hidden states, then compute MBE on prefixes of increasing length using `OnlineMBE`.

In [ ]:
def extract_gold_answer(answer_text: str) -> str:
    match = re.search(r"####\s*(.+)", answer_text)
    return match.group(1).strip().replace(",", "") if match else ""

def extract_predicted_answer(text: str) -> str:
    match = re.search(r"####\s*([\d,\.\-]+)", text)
    if match:
        return match.group(1).strip().replace(",", "")
    numbers = re.findall(r"-?[\d,]+\.?\d*", text)
    return numbers[-1].replace(",", "") if numbers else ""

@torch.no_grad()
def generate_rollout(model, tokenizer, question, max_new_tokens=512):
    """Generate a completion using HF model.generate. Returns (text, full_ids, prompt_len)."""
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
    )
    completion_text = tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True)
    return completion_text, output_ids, prompt_len


def batch_generate_vllm(vllm_engine, tokenizer, questions, max_new_tokens=512):
    """Batch-generate completions using vLLM offline engine.
    Returns list of (completion_text, prompt_text) tuples."""
    from vllm import SamplingParams
    sampling_params = SamplingParams(
        temperature=0.7, top_p=0.9, max_tokens=max_new_tokens,
    )
    prompts = []
    for q in questions:
        messages = [{"role": "user", "content": q}]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    outputs = vllm_engine.generate(prompts, sampling_params)
    results = []
    for prompt_text, out in zip(prompts, outputs):
        completion_text = out.outputs[0].text
        results.append((completion_text, prompt_text))
    return results


@torch.no_grad()
def get_hidden_states(model, input_ids):
    """Forward pass returning all hidden states. Returns tuple of (1, T, D) tensors."""
    outputs = model(input_ids, output_hidden_states=True, use_cache=False)
    return outputs.hidden_states


def compute_per_token_mbe(hidden_states, prompt_len):
    """
    Compute cumulative MBE for each token position using OnlineMBE.
    For each prefix length t, returns MBE(hidden[0:t]).
    
    Returns:
        mbe: np.array (n_layers, comp_len)
    """
    n_layers = len(hidden_states) - 1
    seq_len = hidden_states[1].shape[1]
    comp_len = seq_len - prompt_len
    
    if comp_len < 1:
        return np.zeros((n_layers, 0))
    
    mbe = np.zeros((n_layers, comp_len))
    
    for layer_idx in range(n_layers):
        h = hidden_states[layer_idx + 1][0, prompt_len:, :].float()  # (comp_len, D)
        D = h.shape[1]
        
        tracker = OnlineMBE(D, device=h.device)
        for t in range(comp_len):
            tracker.update(h[t])
            mbe[layer_idx, t] = tracker.mbe().item()
    
    return mbe


def build_result_from_completion(model, tokenizer, question, gold_answer, completion_text, prompt_text):
    """Given a pre-generated completion, run HF forward pass for hidden states and compute MBE."""
    full_text = prompt_text + completion_text
    prompt_ids = tokenizer(prompt_text, return_tensors="pt")["input_ids"]
    full_ids = tokenizer(full_text, return_tensors="pt")["input_ids"].to(model.device)
    prompt_len = prompt_ids.shape[1]

    predicted = extract_predicted_answer(completion_text)
    try:
        correct = float(predicted) == float(gold_answer)
    except (ValueError, TypeError):
        correct = False

    hidden_states = get_hidden_states(model, full_ids)
    mbe = compute_per_token_mbe(hidden_states, prompt_len)

    comp_ids = full_ids[0, prompt_len:].tolist()
    tokens = [tokenizer.decode([tid]) for tid in comp_ids]

    return {
        "question": question[:100],
        "gold": gold_answer,
        "predicted": predicted,
        "correct": correct,
        "comp_len": full_ids.shape[1] - prompt_len,
        "completion_text": completion_text[:200],
        "tokens": tokens,
        "mbe": mbe,
    }


def run_single_example(model, tokenizer, question, gold_answer, max_new_tokens=512):
    """Run a full rollout (HF generate + MBE) for a single example."""
    completion_text, output_ids, prompt_len = generate_rollout(model, tokenizer, question, max_new_tokens)

    predicted = extract_predicted_answer(completion_text)
    try:
        correct = float(predicted) == float(gold_answer)
    except (ValueError, TypeError):
        correct = False

    hidden_states = get_hidden_states(model, output_ids)
    mbe = compute_per_token_mbe(hidden_states, prompt_len)

    comp_ids = output_ids[0, prompt_len:].tolist()
    tokens = [tokenizer.decode([tid]) for tid in comp_ids]

    return {
        "question": question[:100],
        "gold": gold_answer,
        "predicted": predicted,
        "correct": correct,
        "comp_len": output_ids.shape[1] - prompt_len,
        "completion_text": completion_text[:200],
        "tokens": tokens,
        "mbe": mbe,
    }

print("Helpers ready.")

Helpers ready.


## 3. Run Rollouts
Generate completions for a few GSM8K problems and compute per-layer MBE dynamics.

In [ ]:
torch.manual_seed(42)
indices = torch.randperm(len(dataset))[:N_SAMPLES].tolist()

examples = [(dataset[idx]["question"], extract_gold_answer(dataset[idx]["answer"])) for idx in indices]

if USE_VLLM and vllm_engine is not None:
    # ---- Fast path: vLLM batch generation, then HF forward for hidden states ----
    import time
    t0 = time.time()
    questions = [q for q, _ in examples]
    vllm_outputs = batch_generate_vllm(vllm_engine, tokenizer, questions, MAX_NEW_TOKENS)
    t_gen = time.time() - t0
    print(f"vLLM batch generation: {len(questions)} samples in {t_gen:.1f}s ({t_gen/len(questions):.2f}s/sample)")

    results = []
    for i, ((question, gold), (completion_text, prompt_text)) in enumerate(zip(examples, vllm_outputs)):
        r = build_result_from_completion(model, tokenizer, question, gold, completion_text, prompt_text)
        results.append(r)
        tag = "✓" if r["correct"] else "✗"
        print(f"[{i+1}/{N_SAMPLES}] {tag}  pred={r['predicted']:>8s}  gold={r['gold']:>8s}  len={r['comp_len']}")
else:
    # ---- Fallback: HF generate one at a time ----
    results = []
    for i, (question, gold) in enumerate(examples):
        r = run_single_example(model, tokenizer, question, gold, MAX_NEW_TOKENS)
        results.append(r)
        tag = "✓" if r["correct"] else "✗"
        print(f"[{i+1}/{N_SAMPLES}] {tag}  pred={r['predicted']:>8s}  gold={r['gold']:>8s}  len={r['comp_len']}")

correct_results = [r for r in results if r["correct"]]
incorrect_results = [r for r in results if not r["correct"]]
print(f"\nCorrect: {len(correct_results)}  |  Incorrect: {len(incorrect_results)}")

[1/6] ✗  pred=          gold=      33  len=512
[2/6] ✗  pred=    3.00  gold=      57  len=512
[3/6] ✗  pred=       3  gold=       7  len=512
[4/6] ✗  pred=          gold=      48  len=512
[5/6] ✗  pred=      16  gold=    5600  len=512
[6/6] ✗  pred=          gold=      60  len=512

Correct: 0  |  Incorrect: 6


## 4. Visualize MBE Dynamics

### 4a. Single-rollout heatmaps
For each rollout, show **Cumulative MBE** as a layer × prefix length heatmap.

In [ ]:
def plot_heatmap(result, fig=None):
    """Plot Cumulative MBE heatmap (layer × prefix length) for a single rollout."""
    if fig is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)
    else:
        ax = fig.subplots(1, 1)
    
    comp_len = result["comp_len"]
    positions = np.arange(1, comp_len + 1)
    tag = "✓ Correct" if result["correct"] else "✗ Incorrect"
    info = f"{tag}  |  gold={result['gold']}  pred={result['predicted']}  |  len={comp_len}"
    
    mbe = result["mbe"]
    n_layers = mbe.shape[0]
    if mbe.shape[1] == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        return
    
    im = ax.imshow(mbe, aspect="auto", cmap="YlOrRd", interpolation="nearest", origin="lower")
    n_xticks = min(12, comp_len)
    xtick_idx = np.linspace(0, comp_len - 1, n_xticks, dtype=int)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels([positions[i] for i in xtick_idx], fontsize=8)
    ax.set_xlabel("Token position")
    ax.set_yticks(range(0, n_layers, max(1, n_layers // 8)))
    ax.set_ylabel("Layer")
    ax.set_title("Cumulative MBE", fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    fig.suptitle(info, fontsize=12)

for r in results:
    fig = plt.figure(figsize=(10, 5), constrained_layout=True)
    plot_heatmap(r, fig=fig)
plt.show()

KeyError: 'full_mbe'

### 4b. Per-layer MBE trajectory (selected layers)
Show cumulative MBE vs token position for representative layers.

In [ ]:
def plot_trajectories(result, layers=None):
    """Plot cumulative MBE vs token position for selected layers."""
    comp_len = result["comp_len"]
    positions = np.arange(1, comp_len + 1)
    n_layers = result["mbe"].shape[0]
    
    if comp_len == 0:
        return
    
    if layers is None:
        layers = sorted(set([
            0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
        ]))
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)
    tag = "✓" if result["correct"] else "✗"
    
    cmap = plt.cm.viridis
    for i, layer in enumerate(layers):
        color = cmap(i / max(1, len(layers) - 1))
        ax.plot(positions, result["mbe"][layer], label=f"Layer {layer + 1}", color=color, linewidth=1.5)
    ax.set_xlabel("Token position")
    ax.set_ylabel("Cumulative MBE")
    ax.set_title(f"{tag}  gold={result['gold']}  pred={result['predicted']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for r in results:
    plot_trajectories(r)
plt.show()

### 4c. MBE rate of change (ΔMBE)
Show how fast cumulative MBE is growing at each position.

In [ ]:
def plot_rate(result, layers=None):
    """Plot ΔMBE rate for selected layers."""
    comp_len = result["comp_len"]
    n_layers = result["mbe"].shape[0]
    
    if comp_len < 2:
        return
    
    if layers is None:
        layers = sorted(set([
            0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
        ]))
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)
    mid_positions = np.arange(1, comp_len)
    tag = "✓" if result["correct"] else "✗"
    
    cmap = plt.cm.viridis
    for i, layer in enumerate(layers):
        delta = np.diff(result["mbe"][layer])
        color = cmap(i / max(1, len(layers) - 1))
        ax.plot(mid_positions, delta, label=f"Layer {layer + 1}", color=color, linewidth=1, alpha=0.8)
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Token position")
    ax.set_ylabel("ΔMBE")
    ax.set_title(f"{tag}  ΔMBE rate  |  gold={result['gold']}  pred={result['predicted']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

n_show = min(3, len(results))
for r in results[:n_show]:
    plot_rate(r)
plt.show()

## 5. Correct vs Incorrect: Averaged MBE Dynamics
Average the cumulative MBE trajectories across correct and incorrect rollouts.
Interpolate to a common normalized position axis (0–100% of completion).

In [ ]:
def interpolate_mbe(mbe_matrix, n_points=50):
    """Interpolate a per-token MBE matrix to a common normalized axis [0, 1]."""
    comp_len = mbe_matrix.shape[1]
    if comp_len < 2:
        return None
    
    norm_pos = np.linspace(0, 1, comp_len)
    common_axis = np.linspace(0, 1, n_points)
    
    n_layers = mbe_matrix.shape[0]
    interp = np.zeros((n_layers, n_points))
    for layer in range(n_layers):
        interp[layer] = np.interp(common_axis, norm_pos, mbe_matrix[layer])
    return interp

N_POINTS = 50
common_axis = np.linspace(0, 100, N_POINTS)  # 0–100% of completion

def gather_interp(result_list):
    out = [interpolate_mbe(r["mbe"], N_POINTS) for r in result_list]
    return [x for x in out if x is not None]

interp_correct = gather_interp(correct_results)
interp_incorrect = gather_interp(incorrect_results)

n_layers = results[0]["mbe"].shape[0]
print(f"Cumulative MBE  correct={len(interp_correct)}  incorrect={len(interp_incorrect)}")

In [ ]:
# --- 5a. Averaged MBE heatmaps: Correct vs Incorrect ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

has_c = len(interp_correct) > 0
has_i = len(interp_incorrect) > 0

c_mean = np.mean(interp_correct, axis=0) if has_c else None
i_mean = np.mean(interp_incorrect, axis=0) if has_i else None

vmin = min(c_mean.min() if has_c else 999, i_mean.min() if has_i else 999)
vmax = max(c_mean.max() if has_c else 0, i_mean.max() if has_i else 0)

n_xticks = 6
xtick_idx = np.linspace(0, N_POINTS - 1, n_xticks, dtype=int)

for col, (data_mean, clabel, n) in enumerate([
    (c_mean, "Correct", len(interp_correct)),
    (i_mean, "Incorrect", len(interp_incorrect)),
]):
    ax = axes[col]
    if data_mean is not None:
        im = ax.imshow(data_mean, aspect="auto", cmap="YlOrRd", origin="lower",
                      vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8)
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels([f"{common_axis[i]:.0f}" for i in xtick_idx])
    ax.set_xlabel("Completion %")
    ax.set_ylabel("Layer")
    ax.set_title(f"Cumulative MBE — {clabel} (n={n})")

fig.suptitle("Averaged Cumulative MBE Dynamics", fontsize=14)
plt.show()

In [ ]:
# --- 5b. Per-layer averaged trajectories: Correct vs Incorrect ---
layers_to_show = sorted(set([
    0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1
]))

fig, axes = plt.subplots(1, len(layers_to_show),
                         figsize=(5 * len(layers_to_show), 4),
                         constrained_layout=True)

has_c = len(interp_correct) > 0
has_i = len(interp_incorrect) > 0
c_mean = np.mean(interp_correct, axis=0) if has_c else None
i_mean = np.mean(interp_incorrect, axis=0) if has_i else None

for col_i, layer in enumerate(layers_to_show):
    ax = axes[col_i]
    if has_c:
        ax.plot(common_axis, c_mean[layer], color="#27ae60", linewidth=2, label="Correct")
        if len(interp_correct) > 1:
            c_std = np.std([x[layer] for x in interp_correct], axis=0)
            ax.fill_between(common_axis, c_mean[layer] - c_std,
                           c_mean[layer] + c_std, color="#27ae60", alpha=0.15)
    if has_i:
        ax.plot(common_axis, i_mean[layer], color="#e74c3c", linewidth=2, label="Incorrect")
        if len(interp_incorrect) > 1:
            i_std = np.std([x[layer] for x in interp_incorrect], axis=0)
            ax.fill_between(common_axis, i_mean[layer] - i_std,
                           i_mean[layer] + i_std, color="#e74c3c", alpha=0.15)
    
    ax.set_title(f"Layer {layer + 1}", fontsize=11)
    ax.set_xlabel("Completion %")
    if col_i == 0:
        ax.set_ylabel("Cumulative MBE")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle("Cumulative MBE by Layer: Correct vs Incorrect", fontsize=14, y=1.02)
plt.show()

### 5c. Delta heatmap (Correct − Incorrect)
Where in the layer × position space does cumulative MBE differ most?

In [ ]:
# --- 5c. Delta heatmap: Correct − Incorrect ---
fig, ax = plt.subplots(1, 1, figsize=(10, 6), constrained_layout=True)

if len(interp_correct) > 0 and len(interp_incorrect) > 0:
    c_mean = np.mean(interp_correct, axis=0)
    i_mean = np.mean(interp_incorrect, axis=0)
    delta = c_mean - i_mean
    abs_max = max(abs(delta.min()), abs(delta.max()), 1e-6)
    
    im = ax.imshow(delta, aspect="auto", cmap="RdBu_r", origin="lower",
                   vmin=-abs_max, vmax=abs_max)
    n_xticks = 6
    xtick_idx = np.linspace(0, N_POINTS - 1, n_xticks, dtype=int)
    ax.set_xticks(xtick_idx)
    ax.set_xticklabels([f"{common_axis[i]:.0f}%" for i in xtick_idx])
    ax.set_xlabel("Completion %")
    ax.set_ylabel("Layer")
    plt.colorbar(im, ax=ax, shrink=0.8)
else:
    ax.text(0.5, 0.5, "Need both correct & incorrect", ha="center", va="center",
            transform=ax.transAxes)
ax.set_title("ΔMBE (Correct − Incorrect)\nRed = correct > incorrect")

fig.suptitle("ΔMBE: Cumulative MBE", fontsize=14)
plt.show()

## 6. Summary: MBE Growth Profile
For each layer, compute total MBE growth (final − initial) and half-life position.

In [ ]:
# --- 6. MBE growth profile per layer ---
def compute_growth_profile(mbe_matrix):
    """For each layer, compute total growth and half-life position."""
    n_layers, comp_len = mbe_matrix.shape
    growth = []
    half_life = []
    
    for layer in range(n_layers):
        trace = mbe_matrix[layer]
        if comp_len < 2:
            growth.append(0.0)
            half_life.append(0.0)
            continue
        
        total = trace[-1] - trace[0]
        growth.append(total)
        
        mid = trace[0] + total * 0.5
        crossed = np.where(trace >= mid)[0]
        if len(crossed) > 0:
            half_life.append(crossed[0] / (comp_len - 1) * 100)
        else:
            half_life.append(100.0)
    
    return np.array(growth), np.array(half_life)

fig, (ax_growth, ax_hl) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
layers_arr = np.arange(1, n_layers + 1)

for group, color, label in [
    (correct_results, "#27ae60", "Correct"),
    (incorrect_results, "#e74c3c", "Incorrect"),
]:
    valid = [r for r in group if r["comp_len"] > 1]
    if not valid:
        continue
    profiles = [compute_growth_profile(r["mbe"]) for r in valid]
    g_mean = np.mean([p[0] for p in profiles], axis=0)
    hl_mean = np.mean([p[1] for p in profiles], axis=0)
    ax_growth.plot(layers_arr, g_mean, color=color, linewidth=2, label=label, marker="o", markersize=3)
    ax_hl.plot(layers_arr, hl_mean, color=color, linewidth=2, label=label, marker="o", markersize=3)

ax_growth.set_xlabel("Layer")
ax_growth.set_ylabel("MBE growth (final − initial)")
ax_growth.set_title("Total Growth per Layer")
ax_growth.legend()
ax_growth.grid(True, alpha=0.3)

ax_hl.set_xlabel("Layer")
ax_hl.set_ylabel("Half-life (% of completion)")
ax_hl.set_title("Half-Life per Layer")
ax_hl.legend()
ax_hl.grid(True, alpha=0.3)

fig.suptitle("Cumulative MBE Growth Profile", fontsize=14, y=1.02)
plt.show()

In [ ]:
## 7. Text-Aligned MBE Dynamics
Visualize the decoded tokens alongside the cumulative MBE trajectory so you can see **which tokens** correspond to MBE increases, plateaus, or drops.

- **Top**: MBE line plot for a selected layer
- **Bottom**: Colored text strip where each token's background reflects ΔMBE (red = MBE jump, blue = MBE drop)

In [ ]:
def plot_text_mbe(result, layer_idx=-1, window=80, offset=0):
    """
    Visualize decoded tokens alongside cumulative MBE for a single rollout.
    
    Args:
        result:    rollout dict with 'tokens', 'mbe'
        layer_idx: which layer to show (-1 = last)
        window:    how many tokens to show per page
        offset:    starting token index
    """
    tokens = result["tokens"]
    mbe = result["mbe"]
    n_layers = mbe.shape[0]
    if layer_idx < 0:
        layer_idx = n_layers + layer_idx
    
    comp_len = len(tokens)
    end = min(offset + window, comp_len)
    tok_slice = tokens[offset:end]
    mbe_slice = mbe[layer_idx, offset:end]
    
    # Compute per-token ΔMBE
    if offset == 0:
        delta = np.concatenate([[0.0], np.diff(mbe[layer_idx])])
    else:
        delta = np.diff(mbe[layer_idx, max(0, offset-1):end])
        if offset == 0:
            delta = np.concatenate([[0.0], delta])
    delta_slice = delta[:len(tok_slice)]
    
    # --- Layout: MBE trajectory on top, colored text below ---
    fig = plt.figure(figsize=(18, 6))
    gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)
    
    ax_line = fig.add_subplot(gs[0])
    ax_text = fig.add_subplot(gs[1], sharex=ax_line)
    
    x = np.arange(len(tok_slice))
    
    # --- Top: MBE trajectory ---
    ax_line.plot(x, mbe_slice, color="#2c3e50", linewidth=1.5)
    ax_line.fill_between(x, mbe_slice, alpha=0.15, color="#3498db")
    
    # Highlight large ΔMBE jumps
    threshold = np.percentile(np.abs(delta_slice), 90)
    jumps = np.where(np.abs(delta_slice) > threshold)[0]
    for j in jumps:
        color = "#e74c3c" if delta_slice[j] > 0 else "#3498db"
        ax_line.axvline(j, color=color, alpha=0.3, linewidth=1)
    
    tag = "✓" if result["correct"] else "✗"
    ax_line.set_ylabel(f"MBE (Layer {layer_idx + 1})")
    ax_line.set_title(
        f"{tag}  Cumulative MBE  |  Layer {layer_idx + 1}  |  "
        f"tokens {offset}–{end}  |  gold={result['gold']}  pred={result['predicted']}",
        fontsize=11
    )
    ax_line.grid(True, alpha=0.2)
    plt.setp(ax_line.get_xticklabels(), visible=False)
    
    # --- Bottom: colored text strip ---
    abs_max = max(np.abs(delta_slice).max(), 1e-8)
    norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
    cmap = plt.cm.RdBu_r
    
    for i, (tok, d) in enumerate(zip(tok_slice, delta_slice)):
        bg = cmap(norm(d))
        ax_text.axvspan(i - 0.5, i + 0.5, color=bg, alpha=0.7)
        display = tok.replace("\n", "↵").replace("\t", "→")
        if len(display) > 6:
            display = display[:5] + "…"
        ax_text.text(i, 0.5, display, ha="center", va="center",
                     fontsize=6, fontfamily="monospace", rotation=90,
                     color="black" if abs(d) < abs_max * 0.7 else "white")
    
    ax_text.set_xlim(-0.5, len(tok_slice) - 0.5)
    ax_text.set_ylim(0, 1)
    ax_text.set_yticks([])
    ax_text.set_xlabel("Token position (relative to completion start)")
    ax_text.set_ylabel("ΔMBE")
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax_text, orientation="vertical", shrink=0.8, pad=0.02)
    cbar.set_label("ΔMBE", fontsize=9)
    
    fig.tight_layout()
    return fig

print("Text-aligned MBE visualization ready.")

In [ ]:
# --- 7a. Text + MBE for the first rollout, last layer ---
r = results[0]
plot_text_mbe(r, layer_idx=-1, window=80, offset=0)
plt.show()

In [ ]:
# --- 7b. Browse through the rollout in 80-token windows ---
which_result = 0
start = 80

r = results[which_result]
plot_text_mbe(r, layer_idx=-1, window=80, offset=start)
plt.show()

In [ ]:
# --- 7c. Multi-layer text view: compare early vs late layers ---
r = results[0]
n_layers = r["mbe"].shape[0]
for layer in [0, n_layers // 2, n_layers - 1]:
    plot_text_mbe(r, layer_idx=layer, window=80, offset=0)
plt.show()